# CLIP-FGDI: Running on University GPU Server

This notebook runs the full CLIP-FGDI training pipeline on a remote GPU.

**Workflow:**
1. Verify GPU access & install dependencies
2. Configure dataset paths & hyperparameters
3. Run all 3 training stages
4. Evaluate on test datasets

## 1. Environment Setup

In [ ]:
# ============================================================
# 1a. Verify GPU is available
# ============================================================
!nvidia-smi

In [ ]:
# ============================================================
# 1b. Install required packages
# ============================================================
# If your university server already has PyTorch with CUDA, skip the torch install.
# Uncomment the line below ONLY if PyTorch is not pre-installed:
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

!pip install timm yacs prettytable ftfy regex tqdm scipy

In [ ]:
# ============================================================
# 1c. Verify PyTorch sees the GPU
# ============================================================
import torch
print(f"PyTorch version:  {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version:     {torch.version.cuda}")
    print(f"GPU count:        {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  |  {props.total_mem / 1024**3:.1f} GB VRAM")
else:
    raise RuntimeError(
        "No GPU detected! Make sure you selected a GPU runtime. "
        "If using JupyterHub, check your SLURM allocation or kernel."
    )

## 2. Clone / Navigate to CLIP-FGDI

In [ ]:
import os

# ============================================================
# Option A: Clone the repo (if not already on the server)
# ============================================================
# !git clone https://github.com/Qi5Lei/CLIP-FGDI.git
# os.chdir('CLIP-FGDI')

# ============================================================
# Option B: If the repo is already uploaded to the server,
# set the path below to point to it.
# ============================================================
# REPO_DIR = '/home/<your_username>/CLIP-FGDI'  # <-- EDIT THIS
# os.chdir(REPO_DIR)

print(f"Working directory: {os.getcwd()}")
!ls

## 3. Configuration

Edit the variables below to match your setup.

In [ ]:
# ============================================================
# 3a. GPU selection
# ============================================================
# Set which GPU to use. Use '0' for a single-GPU allocation.
# If SLURM gave you GPU 2, set this to '2', etc.
GPU_ID = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = GPU_ID
print(f"Using GPU: {GPU_ID}")

In [ ]:
# ============================================================
# 3b. Dataset & training configuration
# ============================================================

# >>> EDIT THIS: Path to your ReID datasets on the server <<<
DATA_PATH = '/data/ReidShareDataset'  # e.g. '/home/user/datasets/reid'

# Training hyperparameters (adjust batch_size if you run out of VRAM)
BATCH_SIZE     = 128    # Reduce to 64 or 32 if you get CUDA OOM errors
NUM_WORKERS    = 4      # Number of data loading workers
MODEL          = 'clip'
BACKBONE       = 'ViT-B-16'
LOG_PATH       = 'logs'

# Which protocol to run (choose one)
PROTOCOL = 'protocol_1'  # Options: 'protocol_1', 'protocol_2_C3', 'protocol_2_MS', 
                          #          'protocol_2_M', 'protocol_3_C3', 'protocol_3_MS', 'protocol_3_M'

### Expected Dataset Directory Structure

Your `DATA_PATH` should contain the dataset folders like:

```
DATA_PATH/
├── Market/          (or Market-1501)
├── cuhk_sysu/
├── CUHK02/
├── cuhk03/
├── MSMT17/
├── PRID/
├── GRID/
├── VIPeR/
└── iLIDS/
```

## 4. Initialize Model, Data, and Optimizers

In [ ]:
import sys
import time
import datetime
import warnings
import argparse
from functools import partial

import torch
import torch.nn as nn
import torch.multiprocessing
from torch.cuda import amp

torch.multiprocessing.set_sharing_strategy('file_system')
warnings.filterwarnings("ignore")

# Project imports
sys.path.insert(0, os.getcwd())
from reidutils.meter import AverageMeter
from reidutils.metrics import R1_mAP_eval
from cfgs import *
from reidutils import setup_logger
from models import *
from datasets.build import build_data_loader
from loss import make_loss
from loss.supcontrast import SupConLoss
from solver import create_scheduler, WarmupMultiStepLR, make_optimizer_for_IE, \
    make_optimizer_prompt_domain, make_optimizer_prompt

print("All imports successful!")

In [ ]:
# ============================================================
# 4a. Parse configuration (simulates argparse from command line)
# ============================================================
parser = argparse.ArgumentParser(description='train')
parser_test = argparse.ArgumentParser(description='test')

# Select protocol
protocol_fn = {
    'protocol_1': protocol_1,
    'protocol_2_C3': protocol_2_C3,
    'protocol_2_MS': protocol_2_MS,
    'protocol_2_M': protocol_2_M,
    'protocol_3_C3': protocol_3_C3,
    'protocol_3_MS': protocol_3_MS,
    'protocol_3_M': protocol_3_M,
}[PROTOCOL]

parsertrain, parsertest, logname = protocol_fn(parser, parser_test)

# Build args and override with our notebook config
args_train = parsertrain.parse_args([])
args_test  = parsertest.parse_args([])

# Apply overrides from Section 3
args_train.data_path    = DATA_PATH
args_train.batch_size   = BATCH_SIZE
args_train.num_workers  = NUM_WORKERS
args_train.model        = MODEL
args_train.backbone     = BACKBONE
args_train.log_path     = LOG_PATH
args_train.device_id    = GPU_ID
args_test.data_path     = DATA_PATH
args_test.num_workers   = NUM_WORKERS

print(f"Protocol:         {PROTOCOL}")
print(f"Train datasets:   {args_train.train_datasets}")
print(f"Test datasets:    {args_test.test_datasets}")
print(f"Total classes:    {sum(args_train.classes)}")
print(f"Batch size:       {args_train.batch_size}")
print(f"Data path:        {args_train.data_path}")

In [ ]:
# ============================================================
# 4b. Setup logging
# ============================================================
time_now = str(datetime.datetime.now())[:-7]
log_path = os.path.join(args_train.log_path, logname + '_' + args_train.backbone + '_' + time_now)
os.makedirs(log_path, exist_ok=True)

logger_train = setup_logger(args_train.model + '_' + args_train.backbone + '_train', log_path, if_train=True)
logger_test  = setup_logger(args_train.model + '_' + args_train.backbone + '_test', log_path, if_train=False)

logger_train.info(f"Log saved in: {log_path}")
logger_train.info(f"Training cfgs: {str(args_train)}")
logger_train.info(f"Running protocol: {args_train.train_datasets} -> {args_test.test_datasets}")
print(f"Logs will be saved to: {log_path}")

In [ ]:
# ============================================================
# 4c. Build model
# ============================================================
print("Loading model... (this downloads CLIP weights on first run)")

def get_model(args):
    if args.model == 'ViT':
        model = ViT(img_size=args.size_train,
                    stride_size=16,
                    drop_path_rate=0.1,
                    drop_rate=0.,
                    attn_drop_rate=0.,
                    norm_layer=partial(nn.LayerNorm, eps=1e-6),
                    qkv_bias=True)
        model.load_param(args.pretrain_vit_path)
    elif args.model == 'gfnet':
        model = GFNet(
            img_size=(384, 128),
            patch_size=16, embed_dim=384, depth=19, mlp_ratio=4, drop_path_rate=0.15,
            norm_layer=partial(nn.LayerNorm, eps=1e-6)
        )
        model.load_param(args.pretrain_gfnet_path)
    elif args.model == 'clip':
        model = Clip(sum(args.classes), args, domain_num=len(args.train_datasets), epsilon=args.epsilon)
    else:
        model = ViT(img_size=args.size_train,
                    stride_size=16,
                    drop_path_rate=0.1,
                    drop_rate=0.,
                    attn_drop_rate=0.,
                    norm_layer=partial(nn.LayerNorm, eps=1e-6),
                    qkv_bias=True)
        model.load_param(args.pretrain_path)
    return model

model = get_model(args_train).cuda()
print(f"Model loaded and moved to GPU.")

# Print GPU memory usage
mem_alloc = torch.cuda.memory_allocated() / 1024**3
mem_total = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"GPU Memory: {mem_alloc:.2f} GB / {mem_total:.1f} GB")

In [ ]:
# ============================================================
# 4d. Build data loaders
# ============================================================
print("Building data loaders...")
train_loader_stage1, train_loader_stage2, val_loaders = build_data_loader(args_train, args_test)

print(f"Stage-1 loader:  {len(train_loader_stage1)} batches")
print(f"Stage-2 loader:  {len(train_loader_stage2)} batches")
print(f"Test datasets:   {list(val_loaders.keys())}")

In [ ]:
# ============================================================
# 4e. Build loss, optimizers, schedulers
# ============================================================
criterion = make_loss(sum(args_train.classes))

optimizer_prompt        = make_optimizer_prompt(model, args_train)
optimizer_prompt_domain = make_optimizer_prompt_domain(model, args_train)
optimizer_image_encoder = make_optimizer_for_IE(model, args_train)

scheduler_prompt        = create_scheduler(args_train.prompt_epoch, args_train.prompt_lr, optimizer_prompt)
scheduler_prompt_domain = create_scheduler(args_train.prompt_domain_epoch, args_train.prompt_lr, optimizer_prompt_domain)
scheduler_image_encoder = WarmupMultiStepLR(optimizer_image_encoder, [30, 50], 0.1, 0.1, 10, 'linear')

print("Loss, optimizers, and schedulers created.")

## 5. Training

The training pipeline has **3 stages**:
1. **Stage-Prior** — Trains the image encoder with classification loss
2. **Stage-1** — Trains prompt learners (class prompts + domain prompts)
3. **Stage-2** — Fine-tunes the image encoder with text-image alignment

In [ ]:
# ============================================================
# Training helper functions (from run.py)
# ============================================================

def train_stage_prior(train_loader_stage2, model, criterion, optimizer, scheduler, args_train,
                      logger_train, log_path, epochs=3):
    logger_train.info('start stage-prior training')
    device = 'cuda'
    loss_meter = AverageMeter()
    scaler = amp.GradScaler()

    for epoch in range(1, epochs + 1):
        model.train()
        loss_meter.reset()
        scheduler.step()
        for n_iter, (img, vid, _, _, domain, _) in enumerate(train_loader_stage2):
            optimizer.zero_grad()
            img = img.to(device)
            target = vid.to(device)
            target_cam = None
            target_view = None
            with amp.autocast(enabled=True):
                score, feat, image_features = model(x=img, label=target, cam_label=target_cam,
                                                    view_label=target_view)
                loss = criterion(score, feat, target, None)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_meter.update(loss.item(), img.shape[0])
            torch.cuda.synchronize()
            if (n_iter + 1) % args_train.log_period == 0:
                logger_train.info("STAGE-PRIOR Epoch[{}] Iteration[{}/{}] Loss: {:.3f}, Base Lr: {:.2e}"
                                  .format(epoch, (n_iter + 1), len(train_loader_stage2),
                                          loss_meter.avg, scheduler.get_lr()[0]))

        if epoch % args_train.checkpoint_period == 0:
            torch.save(model.state_dict(),
                       os.path.join(log_path,
                                    args_train.model + '_stageprior_{}.pth'.format(epoch)))


def train_stage1(train_loader_stage1, model, optimizer, scheduler, args_train, logger_train, log_path,
                 get_domain=False, epochs=120, omega=0.01):
    logger_train.info('start stage-1 training')
    device = 'cuda'
    loss_meter = AverageMeter()
    accd_meter = AverageMeter()
    scaler = amp.GradScaler()
    xent = SupConLoss('cuda')
    dc = nn.CrossEntropyLoss()
    image_features = []
    labels = []
    domains = []
    camids = []
    cids = []
    model.eval()
    with torch.no_grad():
        for n_iter, (img, pids, camid, viewids, domain, cid) in enumerate(train_loader_stage1):
            img = img.to(device)
            cid = cid.to(device)
            target = pids.to(device)
            camid = camid.to(device)
            with amp.autocast(enabled=True):
                image_feature = model(img, target, get_image=True)
                for i, domain, camid, img_feat, cid in zip(target, domain, camid, image_feature, cid):
                    labels.append(i)
                    domains.append(domain)
                    cids.append(cid)
                    camids.append(camid)
                    image_features.append(img_feat.cpu())

        labels_list = torch.stack(labels, dim=0).cuda()
        domains_list = torch.stack(domains, dim=0).cuda()
        cids_list = torch.stack(cids, dim=0).cuda()
        image_features_list = torch.stack(image_features, dim=0).cuda()

        batch = args_train.batch_size
        num_image = labels_list.shape[0]
        i_ter = num_image // batch
        del labels, image_features
        torch.cuda.empty_cache()

    for epoch in range(1, epochs + 1):
        model.train(True)
        loss_meter.reset()
        accd_meter.reset()
        scheduler.step(epoch)
        iter_list = torch.arange(num_image).to(device)
        for i in range(i_ter):
            optimizer.zero_grad()
            if i != i_ter:
                b_list = iter_list[i * batch:(i + 1) * batch]
            else:
                b_list = iter_list[i * batch:num_image]
            target = labels_list[b_list]
            domain = domains_list[b_list]
            cid = cids_list[b_list]
            image_feats = image_features_list[b_list]
            with amp.autocast(enabled=True):
                text_features, dcscore = model(label=target, get_text=True, domain=domain, cam_label=cid,
                                               getdomain=get_domain)
                text_features.requires_grad_(True)
                image_feats.requires_grad_(True)
            loss_i2t = xent(image_feats.float(), text_features.float(), target, target)
            loss_t2i = xent(text_features.float(), image_feats.float(), target, target)

            dc_loss = dc(dcscore, domain)
            loss = loss_i2t + loss_t2i + omega * dc_loss

            accd = (dcscore.max(1)[1] == domain).float().mean()
            accd_meter.update(accd, 1)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_meter.update(loss.item(), batch)

            torch.cuda.synchronize()
            if (i + 1) % args_train.log_period == 0:
                logger_train.info(
                    "STAGE1-Epoch[{}] Iteration[{}/{}] Domain_Acc: {:.3f} Loss: {:.3f}, Base Lr: {:.2e}"
                    .format(epoch, (i + 1), len(train_loader_stage1), accd_meter.avg,
                            loss_meter.avg, scheduler._get_lr(epoch)[0]))

    torch.save(model.promptlearner.state_dict(),
               os.path.join(log_path,
                            args_train.model + '_clsctx_' + str(get_domain) + '.pth'))


def train_stage2(train_loader_stage2, model, criterion, optimizer, scheduler, testloaders, args_train,
                 logger_train, logger_test, log_path, epochs=60):
    logger_train.info('start stage-2 training')
    device = 'cuda'
    loss_meter = AverageMeter()
    acc_meter = AverageMeter()
    accd_meter = AverageMeter()
    accc_meter = AverageMeter()
    num_classes = sum(args_train.classes)
    scaler = amp.GradScaler()
    xent = SupConLoss(device)
    dc = nn.CrossEntropyLoss()
    batch = args_train.batch_size
    i_ter = num_classes // batch
    left = num_classes - batch * (num_classes // batch)
    if left != 0:
        i_ter = i_ter + 1
    text_features = []
    model.eval()
    p2d = train_loader_stage2.dataset.p2d
    with torch.no_grad():
        for i in range(i_ter):
            if i + 1 != i_ter:
                l_list = torch.arange(i * batch, (i + 1) * batch)
            else:
                l_list = torch.arange(i * batch, num_classes)
            with amp.autocast(enabled=True):
                text_feature, _ = model(label=l_list, get_text=True)
            text_features.append(text_feature.cpu())
        text_features = torch.cat(text_features, 0).cuda()
    for epoch in range(1, epochs + 1):
        model.train()
        start_time = time.time()
        loss_meter.reset()
        acc_meter.reset()
        accd_meter.reset()
        accc_meter.reset()
        scheduler.step()
        for n_iter, (img, vid, _, _, domain, cid) in enumerate(train_loader_stage2):
            optimizer.zero_grad()
            img = img.to(device)
            target = vid.to(device)
            cid = cid.to(device)
            domain = domain.to(device)
            target_cam = None
            target_view = None
            model.eval()
            with torch.no_grad():
                text_feature_p, _ = model(label=target, get_text=True, domain=domain, getdomain=False)
                text_feature_n, _ = model(label=target, get_text=True, domain=domain, getdomain=True)
            model.train()
            with amp.autocast(enabled=True):
                score, feat, image_feats = model(x=img, label=target, cam_label=target_cam,
                                                    view_label=target_view)
                logits = image_feats @ text_features.t()
                loss = criterion(score, feat, target, logits, image_feats, text_feature_p.clone().detach(),
                                 text_feature_n.clone().detach(), beta=args_train.beta)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            acc = ((score[0] + score[1]).max(1)[1] == target).float().mean()
            loss_meter.update(loss.item(), img.shape[0])
            acc_meter.update(acc, 1)

            torch.cuda.synchronize()
            if (n_iter + 1) % args_train.log_period == 0:
                logger_train.info(
                    "STAGE2-Epoch[{}] Iteration[{}/{}] Loss: {:.3f}, AccD: {:.3f}, AccC: {:.3f}, Acc: {:.3f}, Base Lr: {:.2e}"
                    .format(epoch, (n_iter + 1), len(train_loader_stage2),
                            loss_meter.avg, accd_meter.avg, accc_meter.avg, acc_meter.avg, scheduler.get_lr()[0]))
        if epoch % args_train.checkpoint_period == 0:
            torch.save(model.state_dict(),
                       os.path.join(log_path,
                                    args_train.model + '_stage2_{}.pth'.format(epoch)))

        if epoch % args_train.eval_period == 0:
            test(testloaders, model, logger_test)


def test(testloaders, model, logger_test):
    model.eval()
    maps, r1s, r5s, r10s = [], [], [], []
    for name, val_loader in testloaders.items():
        evaluator = R1_mAP_eval(val_loader[1], max_rank=10, feat_norm=False, reranking=False)
        evaluator.reset()
        logger_test.info("Validation Results of {}: ".format(name))
        for n_iter, (img, pids, camids, viewids, domain, cid) in enumerate(val_loader[0]):
            with torch.no_grad():
                img = img.cuda()
                feat = model(img)
                evaluator.update((feat, pids, camids))
        cmc, mAP, _, _, _, _, _ = evaluator.compute()
        logger_test.info("mAP: {:.1%}".format(mAP))
        for r in [1, 5, 10]:
            logger_test.info("CMC curve, Rank-{:<3}:{:.1%}".format(r, cmc[r - 1]))
        logger_test.info("-" * 30)
        torch.cuda.empty_cache()
        maps.append(mAP)
        r1s.append(cmc[0])
        r5s.append(cmc[4])
        r10s.append(cmc[9])
    logger_test.info("Average Results :")
    logger_test.info("Average, mAP:{:.1%}".format(sum(maps) / len(maps)))
    logger_test.info("Average, Rank-1:{:.1%}".format(sum(r1s) / len(r1s)))
    logger_test.info("Average, Rank-5:{:.1%}".format(sum(r5s) / len(r5s)))
    logger_test.info("Average, Rank-10:{:.1%}".format(sum(r10s) / len(r10s)))

print("Training functions defined.")

### 5a. Stage-Prior Training

In [ ]:
%%time
print("=" * 60)
print("STAGE-PRIOR: Training image encoder (warm-up)")
print("=" * 60)

train_stage_prior(
    train_loader_stage2=train_loader_stage2,
    model=model,
    criterion=criterion,
    optimizer=optimizer_image_encoder,
    scheduler=scheduler_image_encoder,
    args_train=args_train,
    logger_train=logger_train,
    log_path=log_path,
    epochs=args_train.prior_epoch
)
print("Stage-Prior complete!")

### 5b. Stage-1: Prompt Learning (Class Prompts)

In [ ]:
%%time
print("=" * 60)
print("STAGE-1a: Training class prompt learner")
print("=" * 60)

train_stage1(
    train_loader_stage1=train_loader_stage1,
    model=model,
    optimizer=optimizer_prompt,
    scheduler=scheduler_prompt,
    args_train=args_train,
    logger_train=logger_train,
    log_path=log_path,
    get_domain=False,
    epochs=args_train.prompt_epoch
)
print("Stage-1a (class prompts) complete!")

In [ ]:
%%time
print("=" * 60)
print("STAGE-1b: Training domain prompt learner")
print("=" * 60)

train_stage1(
    train_loader_stage1=train_loader_stage1,
    model=model,
    optimizer=optimizer_prompt_domain,
    scheduler=scheduler_prompt_domain,
    args_train=args_train,
    logger_train=logger_train,
    log_path=log_path,
    get_domain=True,
    epochs=args_train.prompt_domain_epoch
)
print("Stage-1b (domain prompts) complete!")

### 5c. Stage-2: Image Encoder Fine-Tuning

In [ ]:
%%time
print("=" * 60)
print("STAGE-2: Fine-tuning image encoder with text alignment")
print("=" * 60)

train_stage2(
    train_loader_stage2=train_loader_stage2,
    model=model,
    criterion=criterion,
    optimizer=optimizer_image_encoder,
    scheduler=scheduler_image_encoder,
    testloaders=val_loaders,
    args_train=args_train,
    logger_train=logger_train,
    logger_test=logger_test,
    log_path=log_path,
    epochs=args_train.image_encoder_epoch
)
print("Stage-2 complete!")

## 6. Final Evaluation

In [ ]:
print("=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

test(val_loaders, model, logger_test)

print("\nDone! Check the log files for detailed results:")
print(f"  {log_path}")

## 7. Save Final Model (Optional)

In [ ]:
# Save the final trained model
final_path = os.path.join(log_path, f'{args_train.model}_final.pth')
torch.save(model.state_dict(), final_path)
print(f"Final model saved to: {final_path}")

## Troubleshooting

| Problem | Solution |
| :--- | :--- |
| **CUDA out of memory** | Reduce `BATCH_SIZE` to 64 or 32 in Section 3 |
| **No GPU detected** | Check your SLURM allocation or JupyterHub kernel |
| **Dataset not found** | Verify `DATA_PATH` points to the correct location |
| **Import errors** | Make sure you `cd` into the CLIP-FGDI directory (Section 2) |
| **Connection drops** | Use `nohup` or `tmux` on the server, or use `screen` before launching jupyter |